Chargement des données

In [ ]:
import pandas as pd
fake = pd.read_csv("/content/Fake.csv")
true = pd.read_csv("/content/True.csv")

TF-IDF

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

# Split
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df["text"],
    df["label"],
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

# TF-IDF
vectorizer = TfidfVectorizer(
    max_features=5000,
    stop_words="english",
    min_df=1
)

X_train = vectorizer.fit_transform(X_train_text)
X_test = vectorizer.transform(X_test_text)

print("Train documents :", X_train.shape[0])
print("Test documents :", X_test.shape[0])
print("Nombre de features :", X_train.shape[1])
print("Taille vocabulaire :", len(vectorizer.vocabulary_))

print("\nExemple vecteur TF-IDF (train) :")
print(X_train[0].toarray())

Naive Bayes

In [ ]:
from sklearn.naive_bayes import MultinomialNB

# Création du modèle
nb_model = MultinomialNB()

# Entraînement
nb_model.fit(X_train, y_train)

# Prédiction
y_pred_nb = nb_model.predict(X_test)

Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression

# Création du modèle
lr_model = LogisticRegression(max_iter=1000, random_state=42)

# Entraînement
lr_model.fit(X_train, y_train)

# Prédiction
y_pred_lr = lr_model.predict(X_test)

SVM

In [ ]:
from sklearn.svm import LinearSVC

# Création du modèle
svm_model = LinearSVC(random_state=42)

# Entraînement
svm_model.fit(X_train, y_train)

# Prédiction
y_pred_svm = svm_model.predict(X_test)

Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Création du modèle
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)

# Entraînement
rf_model.fit(X_train, y_train)

# Prédiction
y_pred_rf = rf_model.predict(X_test)

Decision Tree

In [ ]:
from sklearn.tree import DecisionTreeClassifier

# Création du modèle
dt_model = DecisionTreeClassifier(random_state=42)

# Entraînement
dt_model.fit(X_train, y_train)

# Prédiction
y_pred_dt = dt_model.predict(X_test)

TUNING

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression

# Grille des hyperparamètres
param_grid_lr = {
    "C": [0.1, 1, 10]
}

# Recherche des meilleurs paramètres
grid_lr = GridSearchCV(
    LogisticRegression(max_iter=1000, random_state=42),
    param_grid=param_grid_lr,
    cv=3,
    scoring="f1",
    n_jobs=-1
)

# Entraînement
grid_lr.fit(X_train, y_train)

# Meilleur modèle
best_lr = grid_lr.best_estimator_

# Prédiction
y_pred_lr_best = best_lr.predict(X_test)

# Affichage du meilleur paramètre
print("Best parameters for Logistic Regression:", grid_lr.best_params_)

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.svm import LinearSVC

# Grille des hyperparamètres
param_grid_svm = {
    "C": [0.1, 1, 10]
}

# Recherche des meilleurs paramètres
grid_svm = GridSearchCV(
    LinearSVC(random_state=42),
    param_grid=param_grid_svm,
    cv=3,
    scoring="f1",
    n_jobs=-1
)

# Entraînement
grid_svm.fit(X_train, y_train)

# Meilleur modèle
best_svm = grid_svm.best_estimator_

# Prédiction
y_pred_svm_best = best_svm.predict(X_test)

# Affichage du meilleur paramètre
print("Best parameters for SVM:", grid_svm.best_params_)

Comparaison avant / après tuning

In [ ]:
from sklearn.metrics import f1_score

print("F1-score LR avant tuning  :", f1_score(y_test, y_pred_lr))
print("F1-score LR après tuning  :", f1_score(y_test, y_pred_lr_best))

print("F1-score SVM avant tuning :", f1_score(y_test, y_pred_svm))
print("F1-score SVM après tuning :", f1_score(y_test, y_pred_svm_best))

Validation croisée sur le meilleur modèle classique

In [ ]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(best_lr, X_train, y_train, cv=5, scoring='f1')

print("Scores validation croisée :", scores)
print("F1-score moyen (CV)       :", scores.mean())

Tableau résultats

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd

results = pd.DataFrame({
    "Model": [
        "Naive Bayes",
        "Logistic Regression",
        "Logistic Regression (Tuned)",
        "SVM",
        "SVM (Tuned)",
        "Random Forest",
        "Decision Tree"
    ],
    "Accuracy": [
        accuracy_score(y_test, y_pred_nb),
        accuracy_score(y_test, y_pred_lr),
        accuracy_score(y_test, y_pred_lr_best),
        accuracy_score(y_test, y_pred_svm),
        accuracy_score(y_test, y_pred_svm_best),
        accuracy_score(y_test, y_pred_rf),
        accuracy_score(y_test, y_pred_dt)
    ],
    "Precision": [
        precision_score(y_test, y_pred_nb),
        precision_score(y_test, y_pred_lr),
        precision_score(y_test, y_pred_lr_best),
        precision_score(y_test, y_pred_svm),
        precision_score(y_test, y_pred_svm_best),
        precision_score(y_test, y_pred_rf),
        precision_score(y_test, y_pred_dt)
    ],
    "Recall": [
        recall_score(y_test, y_pred_nb),
        recall_score(y_test, y_pred_lr),
        recall_score(y_test, y_pred_lr_best),
        recall_score(y_test, y_pred_svm),
        recall_score(y_test, y_pred_svm_best),
        recall_score(y_test, y_pred_rf),
        recall_score(y_test, y_pred_dt)
    ],
    "F1-score": [
        f1_score(y_test, y_pred_nb),
        f1_score(y_test, y_pred_lr),
        f1_score(y_test, y_pred_lr_best),
        f1_score(y_test, y_pred_svm),
        f1_score(y_test, y_pred_svm_best),
        f1_score(y_test, y_pred_rf),
        f1_score(y_test, y_pred_dt)
    ]
})

results = results.sort_values(by="F1-score", ascending=False).reset_index(drop=True)
print(results)

Graphique

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
plt.bar(results["Model"], results["F1-score"])
plt.xticks(rotation=45, ha="right")
plt.title("Comparaison des modèles selon le F1-score")
plt.xlabel("Modèles")
plt.ylabel("F1-score")
plt.tight_layout()
plt.show()

Meilleur modèle

In [ ]:
print("Le meilleur modèle est :", results.iloc[0]["Model"])
print("Son F1-score est       :", results.iloc[0]["F1-score"])

Sauvegarde modèle

In [ ]:
import pickle

best_model = best_lr

with open("model.pkl", "wb") as f:
    pickle.dump(best_model, f)

with open("vectorizer.pkl", "wb") as f:
    pickle.dump(vectorizer, f)

print("model.pkl et vectorizer.pkl ont été sauvegardés.")